In [ ]:
# ============================================================
# PatchCore Backbone Comparison on Lusitano
# Method: PatchCore + Greedy Coreset + Max Image Score
# Dataset: Lusitano_Dataset
# SEED = 42 | No CenterCrop | Local /content dataset copy
#
# Backbones:
#   EfficientNet-B0
#   DenseNet121
#   EfficientNet-B5
#   MobileNetV3-Large
#   ResNet50
#   WideResNet50-2
#
# Output columns include:
#   Backbone | Layers | D | Patch_Grid_N | Backbone_MB | Memory_Bank_MB | Total_MB
#   Peak_GPU_Memory_MB | ComputationTime/Image | End_to_End_Time/Image
#   AUC | AP | F1 | TN | FP | FN | TP | Precision | Recall | Specificity
#   Balanced_Accuracy | Edge_Efficiency | training/build timing
# ============================================================

# ============================================================
# 1) Imports
# ============================================================

import os
import gc
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from PIL import Image, ImageFile

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import (
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_b5, EfficientNet_B5_Weights,
    mobilenet_v3_large, MobileNet_V3_Large_Weights,
    densenet121, DenseNet121_Weights,
    resnet50, ResNet50_Weights,
    wide_resnet50_2, Wide_ResNet50_2_Weights,
)

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    confusion_matrix,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

from google.colab import drive
drive.mount("/content/drive")

# ============================================================
# 2) Reproducibility
# ============================================================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

ImageFile.LOAD_TRUNCATED_IMAGES = True
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

try:
    torch.set_float32_matmul_precision("medium")
except Exception:
    pass

# ============================================================
# 3) Paths and local dataset copy
# ============================================================

DRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/<YOUR_DATASET_FOLDER>/Lusitano_Dataset")
LOCAL_DATASET_ROOT = Path("/content/Lusitano_Dataset")
RECOPY_LOCAL_DATASET = False

def copy_dataset_to_local(drive_root: Path, local_root: Path, recopy=False):
    if not drive_root.exists():
        raise ValueError(f"Drive dataset not found: {drive_root}")

    expected_train = local_root / "nondefects" / "nondefects"
    expected_test = local_root / "test" / "test"

    if recopy and local_root.exists():
        print("Removing old local dataset copy:", local_root)
        shutil.rmtree(local_root)

    if expected_train.exists() and expected_test.exists():
        print("Using existing local dataset copy:", local_root)
        return local_root

    if local_root.exists():
        print("Local dataset exists but expected structure was not found. Removing:", local_root)
        shutil.rmtree(local_root)

    print("Copying dataset from Drive to local Colab storage...")
    print("From:", drive_root)
    print("To  :", local_root)
    start = time.perf_counter()
    shutil.copytree(drive_root, local_root)
    elapsed = time.perf_counter() - start
    print(f"Local copy completed in {elapsed:.2f} sec")
    return local_root

DATASET_ROOT = copy_dataset_to_local(DRIVE_DATASET_ROOT, LOCAL_DATASET_ROOT, recopy=RECOPY_LOCAL_DATASET)

train_good_path = DATASET_ROOT / "nondefects" / "nondefects"
test_root_path  = DATASET_ROOT / "test" / "test"

if not train_good_path.exists():
    raise ValueError(f"Training path not found: {train_good_path}")
if not test_root_path.exists():
    raise ValueError(f"Test path not found: {test_root_path}")

print("Dataset root used:", DATASET_ROOT)
print("Train folder     :", train_good_path)
print("Test folder      :", test_root_path)

# ============================================================
# 4) Fixed PatchCore settings
# ============================================================

METHOD_NAME = "PatchCore + Greedy Coreset + Max Image Score"

IMG_SIZE = 448
BATCH_SIZE = 16
NUM_WORKERS = 0

PATCHES_PER_IMAGE = 200
PRE_POOL = 400_000
MAX_MEM_PATCHES = 20_000

NN_CHUNK = 40_000
CORESET_CHUNK = 40_000
THRESH_SAMPLE_IMAGES = 2000
STREAM_POOL_MARGIN = 50_000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Please enable GPU in Colab: Runtime > Change runtime type > GPU")

print("Using device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# 5) Backbone configurations
# ============================================================

BACKBONE_CONFIGS = [
    {
        "Backbone": "EfficientNet-B0",
        "builder": efficientnet_b0,
        "weights": EfficientNet_B0_Weights.DEFAULT,
        "hook_modules": ["features.3", "features.5", "features.7"],
        "Layers": "features[3], features[5], features[7]",
    },
    {
        "Backbone": "DenseNet121",
        "builder": densenet121,
        "weights": DenseNet121_Weights.DEFAULT,
        "hook_modules": ["features.denseblock1", "features.denseblock2", "features.denseblock3"],
        "Layers": "denseblock1, denseblock2, denseblock3",
    },
    {
        "Backbone": "EfficientNet-B5",
        "builder": efficientnet_b5,
        "weights": EfficientNet_B5_Weights.DEFAULT,
        "hook_modules": ["features.3", "features.5", "features.7"],
        "Layers": "features[3], features[5], features[7]",
    },
    {
        "Backbone": "MobileNetV3-Large",
        "builder": mobilenet_v3_large,
        "weights": MobileNet_V3_Large_Weights.DEFAULT,
        "hook_modules": ["features.6", "features.12", "features.16"],
        "Layers": "features[6], features[12], features[16]",
    },
    {
        "Backbone": "ResNet50",
        "builder": resnet50,
        "weights": ResNet50_Weights.DEFAULT,
        "hook_modules": ["layer1", "layer2", "layer3"],
        "Layers": "layer1, layer2, layer3",
    },
    {
        "Backbone": "WideResNet50-2",
        "builder": wide_resnet50_2,
        "weights": Wide_ResNet50_2_Weights.DEFAULT,
        "hook_modules": ["layer1", "layer2", "layer3"],
        "Layers": "layer1, layer2, layer3",
    },
]

# ============================================================
# 6) Output paths
# ============================================================

SAVE_DIR = Path("/content/drive/MyDrive/<YOUR_OUTPUT_FOLDER>/backbone_comparison_patchcore_greedy_max_lusitano")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

RESULT_CSV = SAVE_DIR / "patchcore_greedy_max_backbone_comparison_lusitano_seed42.csv"
SCORES_CSV = SAVE_DIR / "patchcore_greedy_max_backbone_comparison_lusitano_image_scores_seed42.csv"
BAD_IMAGES_CSV = SAVE_DIR / "patchcore_greedy_max_backbone_comparison_lusitano_bad_images_seed42.csv"

print("Results will be saved to:", RESULT_CSV)

# ============================================================
# 7) Memory and metric helpers
# ============================================================

def bytes_to_mb(x):
    return x / (1024 ** 2)

def tensor_size_mb(tensor):
    return bytes_to_mb(tensor.numel() * tensor.element_size())

def model_size_mb(model):
    total_bytes = 0
    for p in model.parameters():
        total_bytes += p.numel() * p.element_size()
    for b in model.buffers():
        total_bytes += b.numel() * b.element_size()
    return bytes_to_mb(total_bytes)

def reset_peak_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

def peak_gpu_memory_mb():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        return bytes_to_mb(torch.cuda.max_memory_allocated())
    return 0.0

def edge_efficiency_score(auc, ap, f1, time_per_image, total_mb, peak_gpu_mb):
    numerator = 0.25 * auc + 0.35 * ap + 0.40 * f1
    denominator = 0.20 * time_per_image + 0.40 * total_mb + 0.40 * peak_gpu_mb
    if denominator <= 0:
        return 0.0
    return numerator / denominator

# ============================================================
# 8) Transform: no CenterCrop
# ============================================================

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# ============================================================
# 9) Dataset utilities
# ============================================================

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
bad_image_records = []

def list_images(folder: Path, recursive=True):
    paths = []
    iterator = folder.rglob("*") if recursive else folder.glob("*")
    for p in sorted(iterator):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            try:
                with Image.open(p) as img:
                    img.verify()
                paths.append(p)
            except Exception as e:
                print("Corrupted image skipped:", p, e)
                bad_image_records.append({"image_path": str(p), "error": repr(e)})
    return paths

class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = list(paths)
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        p = self.paths[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), str(p)

class TestImageDataset(Dataset):
    def __init__(self, items, transform):
        self.items = list(items)
        self.transform = transform
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        p, label = self.items[idx]
        img = Image.open(p).convert("RGB")
        return self.transform(img), int(label), str(p)

def make_image_loader(paths, batch_size=BATCH_SIZE, shuffle=False):
    return DataLoader(ImagePathDataset(paths, transform), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=False)

def make_test_loader(items, batch_size=BATCH_SIZE):
    return DataLoader(TestImageDataset(items, transform), batch_size=batch_size, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), drop_last=False)

# ============================================================
# 10) Load train/test paths once
# ============================================================

train_paths = list_images(train_good_path)
if len(train_paths) == 0:
    raise ValueError("No training normal images found.")

def get_label(folder_name):
    name = folder_name.lower().replace("_", "-").strip()
    if name == "non-defects":
        return 0
    if name == "defects":
        return 1
    return None

test_items = []
for folder in sorted(test_root_path.iterdir()):
    if not folder.is_dir():
        continue
    label = get_label(folder.name)
    if label is None:
        print("Skipping unknown folder:", folder.name)
        continue
    paths = list_images(folder)
    for p in paths:
        test_items.append((p, label))

if len(test_items) == 0:
    raise ValueError("No test images found.")

test_loader = make_test_loader(test_items)

print("\nDataset count summary")
print("Training normal images:", len(train_paths))
print("Total test images     :", len(test_items))
print("Normal test images    :", sum(1 for _, y in test_items if y == 0))
print("Defect test images    :", sum(1 for _, y in test_items if y == 1))

if bad_image_records:
    pd.DataFrame(bad_image_records).to_csv(BAD_IMAGES_CSV, index=False)
    print("Bad image list saved:", BAD_IMAGES_CSV)

# ============================================================
# 11) Generic feature extractor using hooks
# ============================================================

class HookedFeatureExtractor(torch.nn.Module):
    def __init__(self, builder, weights, hook_modules, backbone_name):
        super().__init__()
        self.backbone_name = backbone_name
        self.hook_modules = list(hook_modules)
        self.model = builder(weights=weights)
        self.model.eval()
        for p in self.model.parameters():
            p.requires_grad = False
        self.features = []
        self.handles = []
        for module_name in self.hook_modules:
            module = self.model.get_submodule(module_name)
            handle = module.register_forward_hook(self._make_hook(module_name))
            self.handles.append(handle)
    def _make_hook(self, module_name):
        def hook(module, input, output):
            self.features.append(output)
        return hook
    @torch.no_grad()
    def forward(self, x):
        self.features = []
        _ = self.model(x)
        if len(self.features) != len(self.hook_modules):
            raise RuntimeError(f"{self.backbone_name}: expected {len(self.hook_modules)} feature maps, got {len(self.features)}")
        fmap_size = min(f.shape[-2] for f in self.features)
        resize = torch.nn.AdaptiveAvgPool2d(fmap_size)
        resized = [resize(f) for f in self.features]
        patch_features = torch.cat(resized, dim=1)
        B, C, H, W = patch_features.shape
        patch_features = patch_features.reshape(B, C, H * W).permute(0, 2, 1)
        return patch_features.contiguous()
    def remove_hooks(self):
        for h in self.handles:
            h.remove()
        self.handles = []

# ============================================================
# 12) Candidate pool extraction with memory-safe streaming cap
# ============================================================

def compact_random_key_pool(feature_chunks, key_chunks, max_size):
    features = torch.cat(feature_chunks, dim=0)
    keys = torch.cat(key_chunks, dim=0)
    if features.shape[0] > max_size:
        keep_idx = torch.topk(keys, k=max_size, largest=False).indices
        features = features[keep_idx].contiguous()
        keys = keys[keep_idx].contiguous()
    return [features], [keys], int(features.shape[0])

@torch.no_grad()
def extract_candidate_pool(backbone, train_paths, desc):
    set_seed(SEED)
    loader = make_image_loader(train_paths, batch_size=BATCH_SIZE, shuffle=False)
    cpu_generator = torch.Generator(device="cpu")
    cpu_generator.manual_seed(SEED + 777)
    feature_chunks, key_chunks = [], []
    total_kept = 0
    total_seen = 0
    print("\n" + desc)
    extraction_start = time.perf_counter()
    for xb, _ in tqdm(loader, desc=desc):
        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)
        B, N, D = feats.shape
        sampled_list = []
        for i in range(B):
            n = min(PATCHES_PER_IMAGE, N)
            idx = torch.randperm(N, device=DEVICE)[:n]
            sampled_list.append(feats[i, idx].detach().float().cpu())
        batch_features = torch.cat(sampled_list, dim=0)
        m = batch_features.shape[0]
        total_seen += m
        batch_keys = torch.rand(m, generator=cpu_generator, dtype=torch.float32)
        feature_chunks.append(batch_features)
        key_chunks.append(batch_keys)
        total_kept += m
        if total_kept > PRE_POOL + STREAM_POOL_MARGIN:
            feature_chunks, key_chunks, total_kept = compact_random_key_pool(feature_chunks, key_chunks, PRE_POOL)
        del xb, feats, sampled_list, batch_features, batch_keys
    feature_chunks, key_chunks, total_kept = compact_random_key_pool(feature_chunks, key_chunks, PRE_POOL)
    candidate_pool = feature_chunks[0]
    extraction_time_sec = time.perf_counter() - extraction_start
    print("Total sampled patches seen:", total_seen)
    print("Final candidate pool      :", tuple(candidate_pool.shape))
    print(f"Candidate extraction time : {extraction_time_sec:.3f} sec")
    return candidate_pool, extraction_time_sec, int(total_seen)

# ============================================================
# 13) Greedy Coreset
# ============================================================

@torch.no_grad()
def greedy_coreset_gpu(features_cpu, max_samples, chunk=40_000, use_fp16=True, device="cuda", seed=42):
    random.seed(seed)
    N, C = features_cpu.shape
    if N <= max_samples:
        return features_cpu.clone()
    feats = features_cpu.to(device, non_blocking=True).contiguous()
    if use_fp16:
        feats = feats.half()
    selected_idx = torch.empty((max_samples,), dtype=torch.long, device=device)
    first = random.randint(0, N - 1)
    selected_idx[0] = first
    center = feats[first:first + 1]
    min_d = torch.empty((N,), device=device, dtype=torch.float32)
    for start in range(0, N, chunk):
        x = feats[start:start + chunk]
        min_d[start:start + chunk] = (x - center).float().pow(2).sum(dim=1)
    for i in tqdm(range(1, max_samples), desc="Greedy coreset"):
        farthest = torch.argmax(min_d).item()
        selected_idx[i] = farthest
        center = feats[farthest:farthest + 1]
        for start in range(0, N, chunk):
            x = feats[start:start + chunk]
            d = (x - center).float().pow(2).sum(dim=1)
            min_d[start:start + chunk] = torch.minimum(min_d[start:start + chunk], d)
    selected = feats[selected_idx].float().cpu()
    del feats, selected_idx, min_d
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return selected

# ============================================================
# 14) PatchCore max image anomaly score
# ============================================================

@torch.no_grad()
def image_anomaly_score(patch_feats_gpu, memory_bank_gpu, chunk_size=40_000):
    x = patch_feats_gpu.float()
    P = x.shape[0]
    min_dist = torch.full((P,), float("inf"), device=DEVICE, dtype=torch.float32)
    x2 = x.pow(2).sum(dim=1, keepdim=True)
    for start in range(0, memory_bank_gpu.shape[0], chunk_size):
        mb = memory_bank_gpu[start:start + chunk_size].float()
        mb2 = mb.pow(2).sum(dim=1).unsqueeze(0)
        d2 = x2 + mb2 - 2.0 * (x @ mb.t())
        d2 = torch.clamp(d2, min=0.0)
        min_dist = torch.minimum(min_dist, d2.min(dim=1).values)
    return min_dist.sqrt().max().item()

# ============================================================
# 15) Threshold and evaluation
# ============================================================

@torch.no_grad()
def compute_threshold(backbone, memory_bank_gpu):
    rng = np.random.default_rng(SEED + 100)
    num_for_thresh = min(THRESH_SAMPLE_IMAGES, len(train_paths))
    thresh_indices = rng.permutation(len(train_paths))[:num_for_thresh]
    thresh_subset = [train_paths[i] for i in thresh_indices]
    thresh_loader = make_image_loader(thresh_subset, batch_size=BATCH_SIZE, shuffle=False)
    threshold_scores = []
    print("\nComputing threshold from normal training subset...")
    start = time.perf_counter()
    for xb, _ in tqdm(thresh_loader, desc="Threshold"):
        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)
        for i in range(feats.shape[0]):
            threshold_scores.append(image_anomaly_score(feats[i], memory_bank_gpu, chunk_size=NN_CHUNK))
        del xb, feats
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    threshold_time_sec = time.perf_counter() - start
    threshold_scores = np.array(threshold_scores, dtype=np.float32)
    threshold = threshold_scores.mean() + 3.0 * threshold_scores.std(ddof=1)
    print(f"Internal threshold: {threshold:.6f}")
    print(f"Threshold time sec: {threshold_time_sec:.3f}")
    return float(threshold), threshold_scores, float(threshold_time_sec)

@torch.no_grad()
def evaluate_test_set(backbone, memory_bank_gpu, threshold, backbone_name, layers_label):
    reset_peak_gpu_memory()
    y_true, y_score, image_paths = [], [], []
    compute_total_time_sec = 0.0
    print("\nEvaluating test set...")
    end_to_end_start = time.perf_counter()
    for xb, yb, paths_batch in tqdm(test_loader, desc="Testing"):
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_start = time.perf_counter()
        xb = xb.to(DEVICE, non_blocking=True)
        feats = backbone(xb)
        for i in range(feats.shape[0]):
            score = image_anomaly_score(feats[i], memory_bank_gpu, chunk_size=NN_CHUNK)
            y_score.append(float(score))
            y_true.append(int(yb[i]))
            image_paths.append(str(paths_batch[i]))
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        compute_total_time_sec += time.perf_counter() - compute_start
        del xb, feats
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    end_to_end_total_time_sec = time.perf_counter() - end_to_end_start
    compute_time_per_image = compute_total_time_sec / max(len(y_true), 1)
    end_to_end_time_per_image = end_to_end_total_time_sec / max(len(y_true), 1)
    peak_memory = peak_gpu_memory_mb()
    y_true = np.array(y_true, dtype=np.int32)
    y_score = np.array(y_score, dtype=np.float32)
    y_pred = (y_score > threshold).astype(np.int32)
    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    specificity = tn / max(tn + fp, 1)
    balanced_acc = balanced_accuracy_score(y_true, y_pred)
    df_scores = pd.DataFrame({
        "Backbone": backbone_name,
        "Layers": layers_label,
        "image_path": image_paths,
        "gt_label": y_true,
        "anomaly_score": y_score,
        "pred_label": y_pred,
    })
    eval_result = {
        "AUC": float(auc),
        "AP": float(ap),
        "F1": float(f1),
        "Precision": float(precision),
        "Recall_Defect": float(recall),
        "Specificity_Normal": float(specificity),
        "Balanced_Accuracy": float(balanced_acc),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Compute_Total_Time_sec": float(compute_total_time_sec),
        "End_To_End_Local_Total_Time_sec": float(end_to_end_total_time_sec),
        "ComputationTime/Image": float(compute_time_per_image),
        "End_to_End_Time/Image": float(end_to_end_time_per_image),
        "Peak_GPU_Memory_MB": float(peak_memory),
    }
    return eval_result, df_scores

# ============================================================
# 16) Completed-row support
# ============================================================

if RESULT_CSV.exists():
    df_existing = pd.read_csv(RESULT_CSV)
    print("\nExisting result file found:")
    display(df_existing)
else:
    df_existing = pd.DataFrame()

completed_backbones = set()
if not df_existing.empty and "Backbone" in df_existing.columns:
    completed_backbones = set(df_existing["Backbone"].astype(str).tolist())

# ============================================================
# 17) Run one backbone
# ============================================================

def run_one_backbone(cfg):
    backbone_name = cfg["Backbone"]
    layers_label = cfg["Layers"]
    print("\n" + "=" * 100)
    print("Running backbone:", backbone_name)
    print("Layers          :", layers_label)
    print("=" * 100)
    set_seed(SEED)
    reset_peak_gpu_memory()
    backbone = HookedFeatureExtractor(cfg["builder"], cfg["weights"], cfg["hook_modules"], backbone_name).to(DEVICE).eval()
    backbone_mb = model_size_mb(backbone)
    with torch.no_grad():
        dummy = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
        dummy_feats = backbone(dummy)
        D = int(dummy_feats.shape[-1])
        patch_grid_n = int(dummy_feats.shape[1])
        del dummy, dummy_feats
    print("Feature dimension D:", D)
    print("Patch grid N       :", patch_grid_n)
    print(f"Backbone MB        : {backbone_mb:.3f}")
    candidate_pool, candidate_time_sec, total_sampled_patches_seen = extract_candidate_pool(backbone, train_paths, desc=f"Candidate pool | {backbone_name}")
    candidate_pool_shape = tuple(candidate_pool.shape)
    print("\nBuilding Greedy Coreset memory bank...")
    coreset_start = time.perf_counter()
    memory_bank_cpu = greedy_coreset_gpu(candidate_pool, max_samples=MAX_MEM_PATCHES, chunk=CORESET_CHUNK, use_fp16=(DEVICE == "cuda"), device=DEVICE, seed=SEED)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    coreset_time_sec = time.perf_counter() - coreset_start
    del candidate_pool
    gc.collect()
    torch.cuda.empty_cache()
    memory_bank_mb = tensor_size_mb(memory_bank_cpu)
    total_mb = backbone_mb + memory_bank_mb
    print("Memory bank shape:", tuple(memory_bank_cpu.shape))
    print(f"Memory bank MB   : {memory_bank_mb:.3f}")
    print(f"Total MB         : {total_mb:.3f}")
    print(f"Coreset time sec : {coreset_time_sec:.3f}")
    memory_bank_gpu = memory_bank_cpu.to(DEVICE, non_blocking=True)
    threshold, threshold_scores, threshold_time_sec = compute_threshold(backbone, memory_bank_gpu)
    eval_result, df_scores = evaluate_test_set(backbone, memory_bank_gpu, threshold, backbone_name, layers_label)
    edge_eff = edge_efficiency_score(eval_result["AUC"], eval_result["AP"], eval_result["F1"], eval_result["ComputationTime/Image"], total_mb, eval_result["Peak_GPU_Memory_MB"])
    result = {
        "Dataset": "Lusitano", "Dataset_Root_Used": str(DATASET_ROOT), "Method": METHOD_NAME,
        "Seed": SEED, "IMG_SIZE": IMG_SIZE, "Batch_Size": BATCH_SIZE, "No_CenterCrop": True,
        "Backbone": backbone_name, "Layers": layers_label, "Hook_Modules": str(cfg["hook_modules"]),
        "D": D, "Patch_Grid_N": patch_grid_n,
        "PATCHES_PER_IMAGE": PATCHES_PER_IMAGE, "PRE_POOL": PRE_POOL, "MAX_MEM_PATCHES": MAX_MEM_PATCHES,
        "Candidate_Pool_Shape": str(candidate_pool_shape), "Total_Sampled_Patches_Seen": int(total_sampled_patches_seen),
        "Backbone_MB": float(backbone_mb), "Memory_Bank_MB": float(memory_bank_mb), "Total_MB": float(total_mb),
        "Peak_GPU_Memory_MB": eval_result["Peak_GPU_Memory_MB"],
        "ComputationTime/Image": eval_result["ComputationTime/Image"],
        "End_to_End_Time/Image": eval_result["End_to_End_Time/Image"],
        "Compute_Total_Time_sec": eval_result["Compute_Total_Time_sec"],
        "End_To_End_Local_Total_Time_sec": eval_result["End_To_End_Local_Total_Time_sec"],
        "Candidate_Extraction_Time_sec": float(candidate_time_sec),
        "Coreset_Build_Time_sec": float(coreset_time_sec),
        "Threshold_Time_sec": float(threshold_time_sec),
        "AUC": eval_result["AUC"], "AP": eval_result["AP"], "F1": eval_result["F1"],
        "Precision": eval_result["Precision"], "Recall_Defect": eval_result["Recall_Defect"],
        "Specificity_Normal": eval_result["Specificity_Normal"], "Balanced_Accuracy": eval_result["Balanced_Accuracy"],
        "Internal_Threshold": float(threshold), "TN": eval_result["TN"], "FP": eval_result["FP"], "FN": eval_result["FN"], "TP": eval_result["TP"],
        "Edge_Efficiency": float(edge_eff),
        "Train_Normal_Count": len(train_paths), "Test_Total_Count": len(test_items),
        "Test_Normal_Count": sum(1 for _, y in test_items if y == 0),
        "Test_Defect_Count": sum(1 for _, y in test_items if y == 1),
    }
    print("\nResult summary")
    print(f"Backbone       : {backbone_name}")
    print(f"Layers         : {layers_label}")
    print(f"AUC            : {result['AUC']:.6f}")
    print(f"AP             : {result['AP']:.6f}")
    print(f"F1             : {result['F1']:.6f}")
    print(f"TN FP FN TP    : {result['TN']} {result['FP']} {result['FN']} {result['TP']}")
    print(f"Compute sec/img: {result['ComputationTime/Image']:.6f}")
    print(f"E2E sec/img    : {result['End_to_End_Time/Image']:.6f}")
    print(f"Footprint MB   : {result['Total_MB']:.3f}")
    print(f"Peak GPU MB    : {result['Peak_GPU_Memory_MB']:.3f}")
    print(f"Edge efficiency: {result['Edge_Efficiency']:.10f}")
    backbone.remove_hooks()
    del backbone, memory_bank_cpu, memory_bank_gpu, threshold_scores
    gc.collect()
    torch.cuda.empty_cache()
    return result, df_scores

# ============================================================
# 18) Run all backbones
# ============================================================

all_new_scores = []
global_start = time.perf_counter()

for cfg in BACKBONE_CONFIGS:
    backbone_name = cfg["Backbone"]
    if backbone_name in completed_backbones:
        print(f"\nSkipping already completed backbone: {backbone_name}")
        continue
    result, df_scores = run_one_backbone(cfg)
    df_new = pd.DataFrame([result])
    if RESULT_CSV.exists():
        df_old = pd.read_csv(RESULT_CSV)
        df_all = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_all = df_new
    max_eff = df_all["Edge_Efficiency"].max()
    df_all["Edge_Efficiency_%"] = (df_all["Edge_Efficiency"] / max_eff) * 100.0 if max_eff > 0 else 0.0
    df_all.to_csv(RESULT_CSV, index=False)
    if SCORES_CSV.exists():
        df_score_old = pd.read_csv(SCORES_CSV)
        df_score_all = pd.concat([df_score_old, df_scores], ignore_index=True)
    else:
        df_score_all = df_scores
    df_score_all.to_csv(SCORES_CSV, index=False)
    print("\nUpdated result CSV saved:")
    print(RESULT_CSV)

global_total_time_sec = time.perf_counter() - global_start

# ============================================================
# 19) Final display
# ============================================================

df_final = pd.read_csv(RESULT_CSV)
max_eff = df_final["Edge_Efficiency"].max()
df_final["Edge_Efficiency_%"] = (df_final["Edge_Efficiency"] / max_eff) * 100.0 if max_eff > 0 else 0.0
df_final.to_csv(RESULT_CSV, index=False)

print("\n" + "=" * 100)
print("FINAL PATCHCORE BACKBONE COMPARISON SUMMARY")
print("=" * 100)

display_columns = [
    "Backbone", "Layers", "D", "Patch_Grid_N", "Backbone_MB", "Memory_Bank_MB", "Total_MB",
    "Peak_GPU_Memory_MB", "ComputationTime/Image", "End_to_End_Time/Image", "AUC", "AP", "F1",
    "Precision", "Recall_Defect", "Specificity_Normal", "Balanced_Accuracy", "TN", "FP", "FN", "TP",
    "Edge_Efficiency", "Edge_Efficiency_%",
]

print("\nAll results")
display(df_final[display_columns])
print("\nSorted by AP")
display(df_final[display_columns].sort_values("AP", ascending=False))
print("\nSorted by F1")
display(df_final[display_columns].sort_values("F1", ascending=False))
print("\nSorted by Edge Efficiency")
display(df_final[display_columns].sort_values("Edge_Efficiency", ascending=False))
print("\nBest by AP")
display(df_final[display_columns].sort_values("AP", ascending=False).head(1))
print("\nBest by F1")
display(df_final[display_columns].sort_values("F1", ascending=False).head(1))
print("\nBest by Edge Efficiency")
display(df_final[display_columns].sort_values("Edge_Efficiency", ascending=False).head(1))

print("\nSaved result CSV:")
print(RESULT_CSV)
print("\nSaved image-level score CSV:")
print(SCORES_CSV)
if bad_image_records:
    print("\nSaved bad image CSV:")
    print(BAD_IMAGES_CSV)
print(f"\nTotal script runtime sec: {global_total_time_sec:.3f}")
print("=" * 100)
